In [6]:
from matplotlib.backend_bases import cursors

from db.db_connection import create_connection
import pandas as pd
from utils.formatter import prettify
import warnings
warnings.filterwarnings("ignore")
connection = create_connection()

query = """SELECT * FROM v_transactions_overview"""
df = pd.read_sql(query, connection)
prettify(df)

,ID,client_id,Customer,asset_id,Asset,Date,transaction_type,net_amount,Amount,Buy Price,Buy Price (EUR),Exchange Fee,Service Fee,Referral Bonus
0,16,1,Anna,2,ETH,2025-04-24,BUY,0.2509,0.2509,1799.84,1582.13,9.33,2.63,0.75
1,40,1,Anna,2,ETH,2025-05-28,BUY,0.2490,0.2490,2687.85,2375.07,6.60,4.97,0.12
2,42,1,Anna,2,ETH,2025-06-02,SELL,-0.0554,0.0554,2522.38,2208.93,2.44,3.40,2.76
3,51,1,Anna,2,ETH,2025-06-13,BUY,0.3249,0.3249,2664.40,2314.45,3.05,3.38,0.49
4,71,1,Anna,2,ETH,2025-07-19,BUY,0.4472,0.4472,3564.39,3059.56,6.82,4.58,0.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
454,404,6,David,1,BTC,2026-04-09,BUY,0.0082,0.0082,71419.65,61010.95,3.30,3.80,2.85
455,414,6,David,1,BTC,2026-04-09,BUY,0.0360,0.0360,71346.08,60948.10,2.72,2.31,2.77
456,434,6,David,1,BTC,2026-04-09,BUY,0.0214,0.0214,71348.50,60950.17,8.42,4.48,1.95
457,450,6,David,1,BTC,2026-04-09,BUY,0.0085,0.0085,71395.18,60990.05,3.71,2.96,1.45


In [7]:
def run_query(query):
    connection=create_connection()
    df = pd.read_sql(query, connection)
    connection.close()
    return df

In [3]:
# Analyse des Portfolio-Werts über die Zeit
# Ziel: prüfen, ob es starke Veränderungen (Spikes) im Portfolio gibt
df= run_query("""SELECT
    snapshot_date,
    SUM(market_value_eur) as total_value
FROM portfolio_daily_snapshot
GROUP BY snapshot_date
ORDER BY snapshot_date DESC
LIMIT 10;""")
df

,snapshot_date,total_value
0,2026-04-09,238126.434841
1,2026-04-08,242540.876389
2,2026-04-06,56538.561356
3,2026-04-04,5163.510360
4,2026-04-01,2221.870889
5,2026-03-31,12228.443264
6,2026-03-30,13542.876935
7,2026-03-28,3478.146804
8,2026-03-22,11254.386851
9,2026-03-21,10349.391768


In [11]:
# Analyse von Käufen und Verkäufen pro Tag
# Ziel: prüfen, ob mehr gekauft als verkauft wurde (Akkumulation)
df = run_query("""SELECT
    transaction_date,
    SUM(CASE WHEN transaction_type = 'BUY' THEN amount ELSE 0 END) AS total_buy,
    SUM(CASE WHEN transaction_type = 'SELL' THEN amount ELSE 0 END) AS total_sell
FROM transactions t
JOIN statuses s ON t.status_id = s.status_id
WHERE s.status_name = 'completed'
GROUP BY transaction_date
ORDER BY transaction_date DESC
LIMIT 15;""")
df

,transaction_date,total_buy,total_sell
0,2026-04-09,7.1461,0.0395
1,2026-04-08,23.2556,0.7655
2,2026-04-06,1.2548,0.0000
3,2026-04-04,0.0000,0.0000
4,2026-04-01,0.0132,0.0000
5,2026-03-31,0.0412,0.0000
6,2026-03-30,0.0000,0.4500
7,2026-03-28,0.0000,0.0208
8,2026-03-22,0.0197,0.0000
9,2026-03-21,0.0000,0.0356


In [8]:
# Analyse der Marktpreise
# Ziel: prüfen, ob Preisänderungen den Portfolio-Wert beeinflussen

df = run_query("""
SELECT
    DATE(price_date) as date,
    asset_id,
    AVG(price_eur) as avg_price_eur
FROM market_prices
GROUP BY DATE(price_date), asset_id
ORDER BY date DESC
LIMIT 10;
""")
df

,date,asset_id,avg_price_eur
0,2026-04-09,1,60607.397785
1,2026-04-09,2,1863.786026
2,2026-04-08,1,61847.204960
3,2026-04-08,2,1936.790073
4,2026-04-06,1,60619.421984
5,2026-04-06,2,1872.786629
6,2026-04-04,1,58082.231275
7,2026-04-04,2,1781.875296
8,2026-04-03,1,58040.559513
9,2026-04-03,2,1784.722572


In [10]:
df = run_query("""
SELECT
    SUM(
        CASE
            WHEN transaction_type = 'BUY' THEN amount
            WHEN transaction_type = 'SELL' THEN -amount
            ELSE 0
        END
    ) AS total_btc
FROM crypto_portfolio_db.transactions
WHERE asset_id = 1
AND status_id = 2;
""")

df

,total_btc
0,3.5756
